# Flight Delay Prediction using XGBoost

This notebook trains an **XGBoost Classifier** to predict whether a flight will be delayed by 15 minutes or more (`ArrDel15`) using flight data from 2018-2022.

### Prerequisites
1. **Runtime:** For faster training, use a GPU runtime in Google Colab:
   * Go to *Runtime* -> *Change runtime type* -> Select **T4 GPU** (or any available GPU).
2. **Data:** The notebook first checks `DATA_DIR` for the `Combined_Flights_*.csv` files. If any are missing, it automatically downloads them from Kaggle using `kagglehub`.
   * This requires a Kaggle API token. Get one from [kaggle.com/settings](https://www.kaggle.com/settings) -> *Create New Token* (downloads `kaggle.json`).
   * In Colab: upload `kaggle.json` and run `os.environ['KAGGLE_USERNAME']`/`KAGGLE_KEY`, or place it at `~/.kaggle/kaggle.json`. `kagglehub` will prompt for credentials on first use if none are found.
   * `pip install kagglehub` if it isn't already installed in your environment.
3. **Memory Optimization:** Since the complete dataset is over 10 GB, we use memory-efficient data types and chunk-based loading with sampling to prevent Out-Of-Memory (OOM) crashes on standard Colab instances (~12.7 GB RAM).

In [ ]:
# Step 1: Import Libraries and Mount Google Drive
# Make sure kagglehub is installed: pip install kagglehub
import os
import gc
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
import xgboost as xgb
import joblib
import kagglehub

# Optional: Mount Google Drive if files are stored in a Drive folder
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except ImportError:
    print('Not running in Google Colab. Skipping Drive mount.')

### Step 2: Configuration & Memory-Efficient Data Loading

Configure the parameters below to control where files are located, whether to use a GPU, and what subset of the data to load.

In [ ]:
# CONFIGURATION
# Adjust DATA_DIR to where your CSV files are saved.
# If uploaded to Google Drive, it might look like '/content/drive/MyDrive/FlightData'
DATA_DIR = '/content' 

# Kaggle dataset slug (owner/dataset-name) used to auto-download files if not found in DATA_DIR
KAGGLE_DATASET = 'robikscube/flight-delay-dataset-20182022'

# Which yearly files to look for / download. Trim this list if you only want a subset of years.
REQUIRED_FILES = [
    'Combined_Flights_2018.csv',
    'Combined_Flights_2019.csv',
    'Combined_Flights_2020.csv',
    'Combined_Flights_2021.csv',
    'Combined_Flights_2022.csv',
]

# Set to True to enable GPU acceleration in XGBoost (highly recommended in Colab with T4 GPU)
USE_GPU = True        

# Set the fraction of data to sample (0.1 means 10%, 1.0 means 100%).
# 0.2 is recommended for standard Colab instances to avoid OOM crashes.
SAMPLE_FRACTION = 0.2 

# If True, predicts delay BEFORE departure (does not use departure delay fields).
# If False, uses departure delay minutes (DepDelayMinutes) as a strong predictor of arrival delay.
PREDICT_PRE_DEPARTURE = True 

# -----------------------------------------------------

# Define specific columns to load (avoids loading unused heavy columns)
cols_to_load = [
    'Month', 'DayofMonth', 'DayOfWeek', 
    'Airline', 'Origin', 'Dest', 
    'CRSDepTime', 'CRSArrTime', 'Distance',
    'Cancelled', 'Diverted', 'ArrDel15'
]

if not PREDICT_PRE_DEPARTURE:
    cols_to_load.append('DepDelayMinutes')

# Define memory-optimized data types
dtypes = {
    'Month': 'int8',
    'DayofMonth': 'int8',
    'DayOfWeek': 'int8',
    'Airline': 'category',  # Categorical types reduce memory usage and work natively in XGBoost
    'Origin': 'category',
    'Dest': 'category',
    'CRSDepTime': 'int16',
    'CRSArrTime': 'int16',
    'Distance': 'float32',
    'Cancelled': 'bool',
    'Diverted': 'bool',
    'ArrDel15': 'float32'  # Stored as float because it can contain NaN values for cancelled/diverted
}

if not PREDICT_PRE_DEPARTURE:
    dtypes['DepDelayMinutes'] = 'float32'

In [ ]:
# Step 2b: Locate CSVs in DATA_DIR, and auto-download from Kaggle via kagglehub if missing
os.makedirs(DATA_DIR, exist_ok=True)

def find_existing_files(directory):
    if not os.path.isdir(directory):
        return []
    found = [f for f in os.listdir(directory) if f.startswith('Combined_Flights_') and f.endswith('.csv')]
    found.sort()
    return found

csv_files = find_existing_files(DATA_DIR)
missing_files = [f for f in REQUIRED_FILES if f not in csv_files]

if missing_files:
    print(f'Missing {len(missing_files)} file(s) in {DATA_DIR}: {missing_files}')
    print(f'Downloading dataset "{KAGGLE_DATASET}" via kagglehub (requires a valid Kaggle API token)...')

    # kagglehub.dataset_download pulls (and caches) the full dataset, returning the local folder path
    kaggle_cache_path = kagglehub.dataset_download(KAGGLE_DATASET)
    print(f'Dataset downloaded/cached at: {kaggle_cache_path}')

    # Copy (or symlink) any required files that aren't already sitting in DATA_DIR
    for fname in REQUIRED_FILES:
        src = os.path.join(kaggle_cache_path, fname)
        dst = os.path.join(DATA_DIR, fname)
        if os.path.exists(dst):
            continue
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f'  Copied {fname} -> {DATA_DIR}')
        else:
            print(f'  WARNING: {fname} not found in downloaded dataset at {kaggle_cache_path}')

    csv_files = find_existing_files(DATA_DIR)
else:
    print(f'All required files already present in {DATA_DIR}. Skipping download.')

if not csv_files:
    print(f'WARNING: No Combined_Flights_*.csv files found in {DATA_DIR}!')
    print('Please check your Kaggle API credentials (kaggle.json) or adjust DATA_DIR/KAGGLE_DATASET.')
else:
    print(f'Using CSV files: {csv_files}')

dfs = []
for file in csv_files:
    file_path = os.path.join(DATA_DIR, file)
    print(f'Loading and cleaning {file}...')
    
    # We load in chunks to prevent reading the entire massive file into memory at once
    chunks = []
    for chunk in pd.read_csv(file_path, usecols=cols_to_load, dtype=dtypes, chunksize=150000):
        # Filter out Cancelled and Diverted flights as we cannot evaluate arrival delays on them
        chunk = chunk[(chunk['Cancelled'] == False) & (chunk['Diverted'] == False)]
        chunk = chunk.drop(columns=['Cancelled', 'Diverted'])
        chunk = chunk.dropna(subset=['ArrDel15'])
        
        if SAMPLE_FRACTION < 1.0:
            chunk = chunk.sample(frac=SAMPLE_FRACTION, random_state=42)
        
        chunks.append(chunk)
        
    df_year = pd.concat(chunks, ignore_index=True)
    print(f'Processed {file} -> shape: {df_year.shape}, memory usage: {df_year.memory_usage().sum() / 1e6:.2f} MB')
    dfs.append(df_year)

if dfs:
    df = pd.concat(dfs, ignore_index=True)
    # Clean up intermediate variables
    del dfs
    gc.collect()
    print(f'\nAll files loaded successfully!')
    print(f'Combined DataFrame shape: {df.shape}')
    print(f'Total memory usage: {df.memory_usage().sum() / 1e6:.2f} MB')
else:
    df = pd.DataFrame()

### Step 3: Feature Preprocessing

We transform time fields (`CRSDepTime` and `CRSArrTime`) into standard hour representations and verify that all categorical fields are properly set.

In [ ]:
if not df.empty:
    # Extract scheduled hour (0-23) from HHMM format
    df['DepHour'] = (df['CRSDepTime'] // 100).astype('int8')
    df['ArrHour'] = (df['CRSArrTime'] // 100).astype('int8')

    # Drop the raw columns to conserve memory
    df = df.drop(columns=['CRSDepTime', 'CRSArrTime'])

    # Ensure categories are parsed correctly
    for col in ['Airline', 'Origin', 'Dest']:
        df[col] = df[col].astype('category')

    print('Preprocessed columns:')
    print(df.dtypes)
    print(f'Class Distribution:\n{df["ArrDel15"].value_counts(normalize=True)}')

### Step 4: Train / Test Split

Separate the features and target variable, then perform a stratified split to maintain the delay ratio in both training and testing datasets.

In [ ]:
if not df.empty:
    # Define X and y
    X = df.drop(columns=['ArrDel15'])
    y = df['ArrDel15'].astype('int8')

    # Train-test split (80-20%)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Further split training into train and validation for early stopping
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
    )

    print(f'Train subset shape: {X_tr.shape}')
    print(f'Validation subset shape: {X_val.shape}')
    print(f'Test set shape: {X_test.shape}')
    
    # Free memory of full combined df to avoid memory leaks
    del df
    gc.collect()

### Step 4b: Hyperparameter Tuning (RandomizedSearchCV)

We search over key XGBoost hyperparameters to find a better-performing configuration than the hand-picked defaults.

**Notes:**
- `RandomizedSearchCV` is used instead of `GridSearchCV` because it samples a fixed number of combinations (`n_iter`) rather than testing every combination, which is far more practical given the dataset size.
- Tuning runs on a **subsample** of the training data (`TUNE_SAMPLE_SIZE`) to keep search time reasonable. The final model in Step 5 is then trained on the **full** training set using the best parameters found here.
- Scaling is intentionally NOT applied. Tree-based models like XGBoost split on feature order/thresholds, not magnitude, so scaling (StandardScaler/MinMaxScaler) has no effect on accuracy here — it would only add an unnecessary preprocessing step.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

if 'X_tr' in locals():
    # Take a manageable subsample purely for the search (final model uses full X_tr)
    TUNE_SAMPLE_SIZE = min(200_000, len(X_tr))
    tune_idx = X_tr.sample(n=TUNE_SAMPLE_SIZE, random_state=42).index
    X_tune, y_tune = X_tr.loc[tune_idx], y_tr.loc[tune_idx]

    print(f'Running RandomizedSearchCV on {TUNE_SAMPLE_SIZE:,} sampled rows...')

    tune_device_params = {'device': 'cuda', 'tree_method': 'hist'} if (USE_GPU and xgb.__version__ >= '2.0.0') else \
                          ({'tree_method': 'gpu_hist'} if USE_GPU else {'tree_method': 'hist'})

    base_model = xgb.XGBClassifier(
        enable_categorical=True,
        random_state=42,
        eval_metric='auc',
        **tune_device_params
    )

    param_distributions = {
        'n_estimators': [200, 300, 500, 700],
        'max_depth': [4, 5, 6, 8, 10],
        'learning_rate': [0.01, 0.03, 0.05, 0.1],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'min_child_weight': [1, 3, 5, 7],
        'gamma': [0, 0.1, 0.2, 0.5],
        'reg_alpha': [0, 0.01, 0.1, 1],
        'reg_lambda': [1, 1.5, 2, 3],
    }

    random_search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_distributions,
        n_iter=25,
        scoring='roc_auc',
        cv=3,
        verbose=2,
        random_state=42,
        n_jobs=1,  # XGBoost GPU training doesn't parallelize well across sklearn's n_jobs; set >1 only if using CPU-only tree_method
    )

    random_search.fit(X_tune, y_tune)

    print('\nBest ROC-AUC (CV):', random_search.best_score_)
    print('Best Parameters:')
    for k, v in random_search.best_params_.items():
        print(f'  {k}: {v}')

### Step 5: Train XGBoost Model

We set up the `XGBClassifier` with GPU acceleration parameters. Native categorical support is enabled via `enable_categorical=True`.

If Step 4b was run, the best parameters found by `RandomizedSearchCV` are automatically used here in place of the hand-picked defaults.

In [ ]:
if 'X_tr' in locals():
    # Detect device settings
    device = 'cuda' if (USE_GPU and xgb.__version__ >= '2.0.0') else ('gpu_hist' if USE_GPU else 'cpu')
    # For modern XGBoost (>=2.0.0), 'cuda' is the device name. For older versions, 'gpu_hist' is used as tree_method.
    
    if USE_GPU:
        print('Configuring XGBoost for GPU training...')
        if xgb.__version__ >= '2.0.0':
            model_params = {
                'device': 'cuda',
                'tree_method': 'hist'
            }
        else:
            model_params = {
                'tree_method': 'gpu_hist'
            }
    else:
        print('Configuring XGBoost for CPU training...')
        model_params = {
            'tree_method': 'hist'
        }

    # Default hyperparameters (used if Step 4b tuning was skipped)
    best_params = {
        'n_estimators': 500,
        'learning_rate': 0.05,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
    }

    # Override with tuned hyperparameters if RandomizedSearchCV was run in Step 4b
    if 'random_search' in locals():
        print('Using tuned hyperparameters from RandomizedSearchCV (Step 4b).')
        best_params = random_search.best_params_
    else:
        print('Step 4b was not run — using default hyperparameters.')

    # Initialize model
    model = xgb.XGBClassifier(
        **best_params,
        enable_categorical=True,  # Enables native handling of pandas 'category' columns
        random_state=42,
        early_stopping_rounds=15, # Stop training if evaluation loss does not improve for 15 rounds
        **model_params
    )
    
    # Train the model
    print('Training started...')
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=50
    )
    print('Training complete!')

### Step 6: Model Evaluation

Evaluate model performance on the hold-out test set using typical classification metrics and plot feature importances.

In [ ]:
if 'model' in locals():
    from sklearn.metrics import accuracy_score

    # Predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Reports
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')
    print('\n--- Classification Report ---')
    print(classification_report(y_test, y_pred))
    
    auc_score = roc_auc_score(y_test, y_pred_proba)
    print(f'ROC-AUC Score: {auc_score:.4f}')
    
    # Visualizations
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    
    # 1. Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax[0])
    ax[0].set_title('Confusion Matrix')
    ax[0].set_xlabel('Predicted Label')
    ax[0].set_ylabel('True Label')
    
    # 2. ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    ax[1].plot(fpr, tpr, label=f'XGBoost (AUC = {auc_score:.4f})', color='darkorange', lw=2)
    ax[1].plot([0, 1], [0, 1], color='navy', linestyle='--')
    ax[1].set_xlabel('False Positive Rate')
    ax[1].set_ylabel('True Positive Rate')
    ax[1].set_title('ROC Curve')
    ax[1].legend(loc='lower right')
    
    plt.tight_layout()
    plt.show()
    
    # Feature Importance Plot
    plt.figure(figsize=(10, 6))
    xgb.plot_importance(model, max_num_features=10, importance_type='gain', ax=plt.gca())
    plt.title('Top Feature Importances (Gain)')
    plt.show()

### Step 7: Save and Download Model

We save the model using both XGBoost's native configuration format (`.json`), which is recommended, and the Python pickle wrapper format (`.joblib`). We then trigger the browser download in Colab.

In [ ]:
if 'model' in locals():
    # 1. Native XGBoost JSON save (highly portable and version-safe)
    model.save_model('flight_delay_xgb.json')
    print('Saved native model to "flight_delay_xgb.json"')
    
    # 2. Standard joblib save
    joblib.dump(model, 'flight_delay_xgb.joblib')
    print('Saved pickle-wrapped model to "flight_delay_xgb.joblib"')
    
    # Download files automatically if running in Google Colab
    try:
        from google.colab import files
        print('\nStarting download to your local machine...')
        files.download('flight_delay_xgb.json')
        files.download('flight_delay_xgb.joblib')
    except ImportError:
        print('\nNot running in Colab. Please download "flight_delay_xgb.json" and "flight_delay_xgb.joblib" manually from the file explorer.')